# 07 · Main experiment: Falcon dual-model scoring (design doc §8.8.12)

**How to run**: pick a Colab server with an **A100 GPU** and click "Run All"; about 40 minutes. If the connection drops, click "Run All" again; finished parts are skipped. Results go to `DATA_DIR/07_falcon_dual_scoring/`.

**What it computes**: the 8,100 continuations and 3,000 human references of notebook 06 are scored with the official setting of the Fast-DetectGPT and Binoculars papers: Falcon-7B-Instruct as the scorer and Falcon-7B as the observer, both 7B models on one A100. Twelve values are stored per position for the zero-shot baselines.

In [ ]:
# 1. Check that the runtime has an A100: the two 7B models need about 30 GB.
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to an A100 GPU"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(name, f"{mem:.0f} GB")
assert mem > 35, f"This {name} has only {mem:.0f} GB, not enough for two 7B models. Choose an A100 GPU."

In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.makedirs("07_falcon_dual_scoring", exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"

In [ ]:
# 3. Install packages and collect the texts to score: each generator's continuations plus the 1,000 human texts with its split.
%pip install -q -U transformers huggingface-hub accelerate
import json, gc, glob, shutil, time
import numpy as np, pandas as pd
GENS = ["Qwen/Qwen2.5-3B", "microsoft/phi-2", "HuggingFaceTB/SmolLM2-1.7B"]
def records_for(gen_name):
    tag = gen_name.split("/")[1]; recs = []
    for f in sorted(glob.glob(f"06_scaled_sweep/gen_{tag}_T*.json")):
        for r in json.load(open(f)):
            recs.append(dict(id=r["id"], domain=r["domain"], model=gen_name, decoding=r["decoding"], who="ai", prefix=r["prefix"], text=r["ai"]))
    for r in json.load(open(f"06_scaled_sweep/humans_{tag}.json")):
        recs.append(dict(id=r["id"], domain=r["domain"], model=gen_name, decoding="human", who="human", prefix=r["prefix"], text=r["human"]))
    return recs
print({g: len(records_for(g)) for g in GENS})

In [ ]:
# 4. Load Falcon-7B-Instruct (scorer) and Falcon-7B (observer); they share a tokenizer.
from transformers import AutoTokenizer, AutoModelForCausalLM
SCORER, OBSERVER, MAX_TOK, BS = "tiiuae/falcon-7b-instruct", "tiiuae/falcon-7b", 512, 8
tok = AutoTokenizer.from_pretrained(SCORER); tok.padding_side = "right"
pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
p_model = AutoModelForCausalLM.from_pretrained(SCORER, dtype=torch.bfloat16).cuda().eval()
q_model = AutoModelForCausalLM.from_pretrained(OBSERVER, dtype=torch.bfloat16).cuda().eval()
print("loaded", f"{torch.cuda.memory_allocated() / 2**30:.1f} GB")

In [ ]:
# 5. Batched scoring. 12 rows per position:
#    0 s_p scorer surprisal, 1 H_p entropy, 2 V_p log-probability variance, 3 top10_p, 4 X=Σ q log p, 5 XV=Σ q (log p)^2 − X^2,
#    6 s_q observer surprisal, 7 rank_p (rank of the actual token under the scorer, 0 = top), 8 cdf_p (total probability of tokens less likely than the actual one),
#    9 rank_q, 10 cdf_q, 11 H_q observer entropy. Continuation only.
@torch.no_grad()
def rank_and_cdf(lp, tgt):
    # lp: [n, vocab] log-probs; tgt: [n]. rank = number of tokens with higher probability; cdf = mass strictly below the observed token.
    obs = lp.gather(1, tgt[:, None])                       # [n, 1]
    rank = (lp > obs).sum(-1).float()
    p = lp.exp()
    cdf = (p * (lp < obs)).sum(-1)
    return rank, cdf

@torch.no_grad()
def score_records(recs, out_path):
    if os.path.exists(out_path):
        print("Already done, skipping:", out_path); return
    enc = []
    for r in recs:
        k = len(tok(r["prefix"])["input_ids"])
        ids = tok(r["prefix"] + r["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) - k >= 10:
            enc.append((r, k, ids))
    enc.sort(key=lambda e: len(e[2]))
    out, t0 = [], time.time()
    for s in range(0, len(enc), BS):
        chunk = enc[s:s + BS]; L = max(len(e[2]) for e in chunk)
        x = torch.full((len(chunk), L), pad); att = torch.zeros((len(chunk), L), dtype=torch.long)
        for i, (_, _, ids) in enumerate(chunk):
            x[i, :len(ids)] = torch.tensor(ids); att[i, :len(ids)] = 1
        x, att = x.cuda(), att.cuda()
        lg_p = p_model(input_ids=x, attention_mask=att).logits
        lg_q = q_model(input_ids=x, attention_mask=att).logits
        for i, (r, k, ids) in enumerate(chunk):
            n = len(ids)
            lp = torch.log_softmax(lg_p[i, :n - 1].float(), -1); lq = torch.log_softmax(lg_q[i, :n - 1].float(), -1)
            tgt = torch.tensor(ids[1:], device=lp.device)
            p = lp.exp(); q = lq.exp()
            e_p = (p * lp).sum(-1); X = (q * lp).sum(-1)
            rank_p, cdf_p = rank_and_cdf(lp, tgt); rank_q, cdf_q = rank_and_cdf(lq, tgt)
            rows = [-lp.gather(1, tgt[:, None])[:, 0], -e_p, (p * lp * lp).sum(-1) - e_p ** 2, p.topk(10, -1).values.sum(-1),
                    X, (q * lp * lp).sum(-1) - X ** 2, -lq.gather(1, tgt[:, None])[:, 0],
                    rank_p, cdf_p, rank_q, cdf_q, -(q * lq).sum(-1)]
            arr = torch.stack(rows).cpu().numpy().astype(np.float32)[:, k - 1:]
            out.append(dict(id=r["id"], domain=r["domain"], model=r["model"], decoding=r["decoding"], who=r["who"], arr=arr))
        if (s // BS) % 50 == 0:
            print(f"  {s}/{len(enc)} {(time.time() - t0) / 60:.1f} min", flush=True)
    pd.DataFrame(out).to_pickle(out_path)
    print("saved", out_path, len(out), f"{(time.time() - t0) / 60:.1f} min")

for gen_name in GENS:
    score_records(records_for(gen_name), f"07_falcon_dual_scoring/scores_falcon_{gen_name.split('/')[1]}.pkl")

In [ ]:
# 6. Quick summary: AUROC of the official dual-model Fast-DetectGPT and Binoculars against the 1,000 human texts with the same split. The full analysis is scripts/e3_falcon_dual.py.
from sklearn.metrics import roc_auc_score
rows = []
for f in sorted(glob.glob("07_falcon_dual_scoring/scores_falcon_*.pkl")):
    s = pd.read_pickle(f)
    s["fdg"] = [(-a[0].sum() - a[4].sum()) / np.sqrt(a[5].sum()) for a in s.arr]     # official: (sum of log p − sum of expectations) / std
    s["bino"] = [-(a[0].mean() / -a[4].mean()) for a in s.arr]                        # negated so that higher = more AI-like
    h = s[s.who == "human"]
    for dec, g in s[s.who == "ai"].groupby("decoding"):
        y = np.r_[np.ones(len(g)), np.zeros(len(h))]
        rows.append(dict(file=f.split("/")[-1], temperature=float(dec[1:]), FastDetectGPT=roc_auc_score(y, np.r_[g.fdg.values, h.fdg.values]),
                         Binoculars=roc_auc_score(y, np.r_[g.bino.values, h.bino.values])))
table = pd.DataFrame(rows).sort_values(["file", "temperature"]); table.to_csv("07_falcon_dual_scoring/summary.csv", index=False)
print(table.round(3).to_string(index=False))

In [ ]:
# 7. Flush unsynced files back to Google Drive.
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/07_falcon_dual_scoring/")